In [ ]:
#Install Dependencies   
%pip install anthropic python-dotenv

In [2]:
# Laod Env Variables    
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
# Create API Client
from anthropic import Anthropic
client = Anthropic()
model = "claude-sonnet-4-6"  # or "claude-sonnet-4-0-20240924" for the latest version

In [ ]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
    }

    if system:
        params["system"] = system

    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    message = client.messages.create(**params)
    return message.content[0].text


In [5]:
# Putting it all together
# Start with an empty message list
messages = []
print(messages)
# Add the initial user question
add_user_message(messages, "Define quantum computing in one sentence")

# Get Claude's response
answer = chat(messages)
print(answer)

# Add Claude's response to the conversation history
add_assistant_message(messages, answer)

# Add a follow-up question
add_user_message(messages, "Write another sentence")

# Get the follow-up response with full context
final_answer = chat(messages)
print(final_answer)
print(messages)

[]
**Quantum computing** is a form of computation that harnesses quantum mechanical phenomena — such as superposition and entanglement — to process information in ways that can solve certain complex problems exponentially faster than classical computers.
Unlike classical computers, which store information as binary bits (0s or 1s), quantum computers use **qubits** that can exist in multiple states simultaneously, enabling them to perform vast numbers of calculations at once.
[{'role': 'user', 'content': 'Define quantum computing in one sentence'}, {'role': 'assistant', 'content': '**Quantum computing** is a form of computation that harnesses quantum mechanical phenomena — such as superposition and entanglement — to process information in ways that can solve certain complex problems exponentially faster than classical computers.'}, {'role': 'user', 'content': 'Write another sentence'}]


In [6]:
messages = []
# Ensure there's at least one message before calling the API
if not messages:
	add_user_message(messages, "Hello")  # seed the conversation

# Without system prompt
answer = chat(messages)
print(answer)

# With system prompt
system = """
You are a patient math tutor.
Do not directly answer a student's questions.
Guide them to a solution step by step.
"""
answer = chat(messages, system=system)
print(answer)



Hello! How are you doing? Is there something I can help you with today? 😊
Hello! 👋 Welcome! I'm here to help guide you through math problems step by step.

Do you have a math problem you'd like to work on today? What are you studying? 😊


In [7]:
# To enable streaming, add stream=True to your messages.create call:
messages = []
add_user_message(messages, "Write a 1 sentence description of a fake database")

stream = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    stream=True
)

for event in stream:
    print(event)

RawMessageStartEvent(message=Message(id='msg_012R434tMABwHEvpF2Tkv75U', container=None, content=[], model='claude-sonnet-4-6', role='assistant', stop_details=None, stop_reason=None, stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='global', input_tokens=18, output_tokens=1, server_tool_use=None, service_tier='standard')), type='message_start')
RawContentBlockStartEvent(content_block=TextBlock(citations=None, text='', type='text'), index=0, type='content_block_start')
RawContentBlockDeltaEvent(delta=TextDelta(text='"', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text='CustomerVault is a fictional relational database containing fabricated records', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text=' of 50,000 imaginary custo

In [8]:
# Simplified Text Streaming

with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages
) as stream:
    for text in stream.text_stream:
        print(text, end="")

"CustomerVault is a fictional relational database containing fabricated records of 10,000 imaginary customers, including made-up names, addresses, purchase histories, and account details, used solely for testing and demonstration purposes."

In [9]:
# Getting the Complete Message

with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages
) as stream:
    for text in stream.text_stream:
        # Send each chunk to your client 
        pass
    
    # Get the complete message for database storage
    final_message = stream.get_final_message()
    
    print (final_message.content[0].text)
    

Here is a 1 sentence description of a fake database:

**"CustomerVault 3.0 is a fictional relational database containing 10,000 mock customer records, including fabricated names, addresses, purchase histories, and account details, designed for software testing and development purposes."**


In [55]:
# Structured data
import json, re

messages = []
add_user_message(messages, "Generate a very short event bridge rule as json")

text = chat(messages, system="Respond with raw JSON only. No markdown, no explanation.")

# Strip markdown fences if present
match = re.search(r'```(?:json)?\s*([\s\S]+?)```', text)
clean_text = match.group(1) if match else text

clean_json = json.loads(clean_text.strip())
print(clean_json)


{'source': ['aws.ec2'], 'detail-type': ['EC2 Instance State-change Notification'], 'detail': {'state': ['running', 'stopped']}}


In [56]:
messages = []

prompt = """
Generate three different sample AWS CLI commands. Each should be very short.
"""
add_user_message(messages, prompt)
text = chat(messages, system="Respond with raw JSON only. No markdown, no explanation.")
text.strip()



'{"commands":["aws s3 ls","aws ec2 describe-instances","aws iam list-users"]}'

In [57]:
from IPython.display import Markdown

Markdown(text)


{"commands":["aws s3 ls","aws ec2 describe-instances","aws iam list-users"]}